In [ ]:
import pandas as pd
import numpy as np
import re

# Raw Input Text Documents and Query
raw_docs = [
    "cat runs behind rat",
    "dog runs behind cat",
    "dog, cat and rat runs behind cheese"
]
raw_query = "cat cheese"

In [ ]:
# Custom stopword list based on your notes
stopwords = ['and', 'is', 'the', 'a', 'in', 'to']

def preprocess(text):
    # Remove punctuation and convert to lowercase
    text = re.sub(r'[^\w\s]', '', text.lower())
    words = text.split()

    # Remove stopwords
    words = [w for w in words if w not in stopwords]

    # Basic Stemming (converting 'runs' to 'run' to match notes exactly)
    words = ['run' if w == 'runs' else w for w in words]
    return words

# Process documents and query
processed_docs = [preprocess(doc) for doc in raw_docs]
processed_query = preprocess(raw_query)

# Build a unique vocabulary list, preserving the order of appearance
vocabulary = []
for doc in processed_docs:
    for word in doc:
        if word not in vocabulary:
            vocabulary.append(word)

print(f"STEP I: Extracted Keywords: {vocabulary}")

STEP I: Extracted Keywords: ['cat', 'run', 'behind', 'rat', 'dog', 'cheese']


In [ ]:
from IPython.display import display
N = len(raw_docs) # Total documents
# Function to compute raw frequencies and normalized frequencies (F_ij)
def get_frequencies(processed_text, vocab):
    raw_freq = np.array([processed_text.count(word) for word in vocab])
    # F_ij = raw_freq / max_freq (preventing division by zero)
    max_f = np.max(raw_freq) if np.max(raw_freq) > 0 else 1
    norm_freq = raw_freq / max_f
    return raw_freq, norm_freq
# Compute frequencies for all docs and query
doc_raw = []
doc_norm = []
for doc in processed_docs:
    raw, norm = get_frequencies(doc, vocabulary)
    doc_raw.append(raw)
    doc_norm.append(norm)
q_raw, q_norm = get_frequencies(processed_query, vocabulary)
# Convert to numpy arrays for matrix math
F_ij = np.array(doc_norm)
F_iq = np.array(q_norm)
# Count how many docs contain each term (n_i)
n_i = np.sum(np.array(doc_raw) > 0, axis=0)
# Build Step II Table (Raw Frequencies)
df_step2 = pd.DataFrame({'Keywords': vocabulary})
for i in range(N):
    df_step2[f'd{i+1}'] = doc_raw[i]
df_step2['q'] = q_raw
print("STEP II: Intra Cluster Similarity (Raw Term Frequencies)")
display(df_step2)

STEP II: Intra Cluster Similarity (Raw Term Frequencies)


,Keywords,d1,d2,d3,q
0,cat,1,1,1,1
1,run,1,1,1,0
2,behind,1,1,1,0
3,rat,1,0,1,0
4,dog,0,1,1,0
5,cheese,0,0,1,1


In [ ]:
# Calculate IDF = log10(1 + (N / n_i))
idf = np.log10(1 + (N / n_i))
# Document weights: w_ij = F_ij * IDF_i
W_ij = F_ij * idf
# Query weights: w_iq = (0.5 + f_iq) * IDF_i
W_iq = (0.5 + F_iq) * idf
# Build Weights Table
df_weights = pd.DataFrame({'Keywords': vocabulary, 'n_i': n_i, 'IDF': idf})
for i in range(N):
    df_weights[f'd{i+1} (w_{i+1}j)'] = W_ij[i]
df_weights['w_iq'] = W_iq
print("STEP III: Inter Cluster Dissimilarity & Base Weights")
display(df_weights.round(3))

STEP III: Inter Cluster Dissimilarity & Base Weights


,Keywords,n_i,IDF,d1 (w_1j),d2 (w_2j),d3 (w_3j),w_iq
0,cat,3,0.301,0.301,0.301,0.301,0.452
1,run,3,0.301,0.301,0.301,0.301,0.151
2,behind,3,0.301,0.301,0.301,0.301,0.151
3,rat,2,0.398,0.398,0.000,0.398,0.199
4,dog,2,0.398,0.000,0.398,0.398,0.199
5,cheese,1,0.602,0.000,0.000,0.602,0.903


In [ ]:
# Build Part 1: Cross Products
df_products = pd.DataFrame({'Keywords': vocabulary})
for i in range(N):
    df_products[f'w{i+1}j * w_iq'] = W_ij[i] * W_iq
# Build Part 2: Squares
df_squares = pd.DataFrame({'Keywords': vocabulary})
for i in range(N):
    df_squares[f'w{i+1}j^2'] = W_ij[i]**2
df_squares['w_iq^2'] = W_iq**2
# Append Sum rows
df_products.loc['Sum'] = ['Sum'] + df_products.iloc[:, 1:].sum().tolist()
df_squares.loc['Sum'] = ['Sum'] + df_squares.iloc[:, 1:].sum().tolist()
print("STEP IV - PART 1: Cross Products (Numerator components)")
display(df_products.round(3))
print("\nSTEP IV - PART 2: Squared Weights (Denominator components)")
display(df_squares.round(3))

STEP IV - PART 1: Cross Products (Numerator components)


,Keywords,w1j * w_iq,w2j * w_iq,w3j * w_iq
0,cat,0.136,0.136,0.136
1,run,0.045,0.045,0.045
2,behind,0.045,0.045,0.045
3,rat,0.079,0.000,0.079
4,dog,0.000,0.079,0.079
5,cheese,0.000,0.000,0.544
Sum,Sum,0.306,0.306,0.929



STEP IV - PART 2: Squared Weights (Denominator components)


,Keywords,w1j^2,w2j^2,w3j^2,w_iq^2
0,cat,0.091,0.091,0.091,0.204
1,run,0.091,0.091,0.091,0.023
2,behind,0.091,0.091,0.091,0.023
3,rat,0.158,0.000,0.158,0.040
4,dog,0.000,0.158,0.158,0.040
5,cheese,0.000,0.000,0.362,0.816
Sum,Sum,0.430,0.430,0.951,1.144


In [ ]:
# Isolate the 'Sum' rows
sums_products = df_products.loc['Sum'].drop('Keywords')
sums_squares = df_squares.loc['Sum'].drop('Keywords')

print("STEP V: Final Cosine Similarity")
print("-" * 35)

# Sim(dj, q) = Sum(w_ij * w_iq) / (sqrt(Sum(w_ij^2)) * sqrt(Sum(w_iq^2)))
for i in range(N):
    num = sums_products[f'w{i+1}j * w_iq']
    den = np.sqrt(sums_squares[f'w{i+1}j^2']) * np.sqrt(sums_squares['w_iq^2'])
    similarity = num / den
    print(f"Sim(d{i+1}, q) = {similarity:.4f}")

STEP V: Final Cosine Similarity
-----------------------------------
Sim(d1, q) = 0.4358
Sim(d2, q) = 0.4358
Sim(d3, q) = 0.8903
